# 03：教学数据准备

按顺序完成：**1. 了解写法 → 2. 审阅 5 条答案 → 3. 检查并导出**。

选择 **Python (Text LoRA - Qwen)** kernel。先阅读 `../datasets/annotation_guide.md`。

本 Notebook 的 5 个新写片段与答案均已由 agent 准备。agent 负责数据编写、修订与检查；你负责监督写法、数据质量和是否继续的决策。审阅通过后导出教学样本。它只离线加载 processor 和训练模板，不加载模型权重、不安装依赖、不启动训练。

In [1]:
import copy
import hashlib
import json
import os
import re
import sys
from dataclasses import asdict
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path
    for candidate in candidates
    for path in (candidate, candidate / "text-lora")
    if (path / "scripts" / "run_inference.py").is_file()
    and (path / "configs" / "models.toml").is_file()
)
sys.path.insert(0, str(project_root / "scripts"))
print("项目：", project_root)
print("解释器：", sys.executable)

项目： /home/muyi086/github/test_LLM/text-lora
解释器： /home/muyi086/github/test_LLM/text-lora/.venv/bin/python3


## 步骤 1：固定目标写法

- 每行 `[角色][语气]正文`，两个标签连续，正文非空。
- 直接台词使用原文姓名或称呼；动作、环境、沉默、内心活动都由旁白叙述。
- 没有明确语气的叙述默认 `[旁白][平静]`；明确低声、大喊等方式优先保留。
- 保留时间、数量、动作次序、否定和指代，不增加身份、动机、声响或情节。
- 只写演绎稿，不加标题、解释、代码围栏、空标签行或 `<think>`。

以下 Cell 读取已经完成的基线，核对固定输入与配置，并复用改写任务的 system 指令。

In [2]:
from check_environment import check_python, load_model_config
from run_inference import load_inference_config, load_prompts, prompts_fingerprint

check_python()
model = load_model_config("qwen3_5_4b")
print("本地模型：", model.path)

eval_cases = load_prompts()
baseline_dir = project_root / "outputs" / "base-20261008T012651_918242Z"
baseline_metadata = json.loads((baseline_dir / "metadata.json").read_text())
assert baseline_metadata["status"] == "complete"
assert baseline_metadata["completed_count"] == baseline_metadata["prompt_count"] == len(eval_cases) == 10
assert baseline_metadata["prompts_sha256"] == prompts_fingerprint(eval_cases)
assert baseline_metadata["inference"] == asdict(load_inference_config())
assert Path(baseline_metadata["model"]["path"]).resolve() == model.path
assert baseline_metadata["model_config_sha256"] == hashlib.sha256((model.path / "config.json").read_bytes()).hexdigest()
TTS_SYSTEM = next(case for case in eval_cases if case["category"] == "tts_rewrite")["messages"][0]["content"]
MAX_LENGTH = 512
print("已核对完整基线：", baseline_dir)
print("训练长度上限：", MAX_LENGTH, "tokens（包含 system、user、assistant 和模板）")
print("沿用 system：", TTS_SYSTEM)

Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3
本地模型： /home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
已核对完整基线： /home/muyi086/github/test_LLM/text-lora/outputs/base-20261008T012651_918242Z
训练长度上限： 512 tokens（包含 system、user、assistant 和模板）
沿用 system： 你是一名有声剧导演。将给定小说片段改写为多角色演绎稿。每行采用[角色][语气]正文，角色沿用原文姓名，无名人物使用原文称呼，叙述使用旁白。保留原文事实、人物关系、动作和悬疑信息，不补充原文没有的情节。只输出演绎稿，不解释改写过程。


## 步骤 2：审阅 5 条答案

按顺序运行“原文”“答案”和“预览”Cell，查看各片段、检查项与对应演绎稿。

5 条答案已经全部编写完整。你可以指出不符合预期的写法，由 agent 修订，无需亲自补写。原文与答案都是新写教学内容，没有使用固定评估片段。

In [6]:
# 原文：保留 id 和 source_id，后续按故事来源拆分训练集与验证集。
SOURCES = [
    {
        "id": "pilot_01_actions",
        "source_id": "pilot_story_01",
        "text": "雨水顺着伞尖滴在门垫上。季禾把伞靠在墙边，低声说：我把文件放在柜子里了。唐悦拉开抽屉，里面只有一张空白纸。",
        "allowed_roles": ["旁白", "季禾", "唐悦"],
        "review_criteria": ["雨滴、靠伞、拉开抽屉均由旁白保留", "季禾台词保留低声", "不混淆柜子与抽屉，保留只有一张空白纸"],
    },
    {
        "id": "pilot_02_three_roles",
        "source_id": "pilot_story_02",
        "text": "许衡把登记簿推到桌子中央。罗薇问：这一页是谁撕掉的？店员说：我接班的时候就这样了。许衡说：先别翻下一页。三人停下了手里的动作。",
        "allowed_roles": ["旁白", "许衡", "罗薇", "店员"],
        "review_criteria": ["罗薇、店员、许衡台词归属正确", "保留推登记簿与三人停下动作", "不解释谁撕掉了这一页"],
    },
    {
        "id": "pilot_03_inner_thought",
        "source_id": "pilot_story_03",
        "text": "程墨看着手里的车票，日期是星期五。他心想，今天明明才星期四。他把车票折好放进口袋，没有把疑惑告诉身旁的人。",
        "allowed_roles": ["旁白"],
        "review_criteria": ["全部由旁白叙述，保留他心想", "保留星期五与星期四", "保留折好车票、放进口袋和没有告诉身旁的人"],
    },
    {
        "id": "pilot_04_no_dialogue",
        "source_id": "pilot_story_04",
        "text": "午后，乔安在阳台整理花盆。最小的那盆土还是湿的，其余几盆已经干了。她把水壶放回原处，没有浇水。楼下传来一声自行车铃。",
        "allowed_roles": ["旁白"],
        "review_criteria": ["全部由旁白叙述，不新增对白", "保留土壤湿干差异与没有浇水", "保留一声自行车铃，不解释谁浇过水"],
    },
    {
        "id": "pilot_05_ambiguity",
        "source_id": "pilot_story_05",
        "text": "值班员看着监控画面说：那个人又来了。白芷问：你认得他？值班员没有回答，只把画面暂停在电梯门打开的那一刻。画面里的人背对镜头，手里拿着一个纸袋。",
        "allowed_roles": ["旁白", "值班员", "白芷"],
        "review_criteria": ["不解释那个人是谁或纸袋里装了什么", "保留没有回答和暂停画面的时刻", "保留背对镜头与手里的一个纸袋"],
    },
]
for source in SOURCES:
    print("\n", source["id"], "来源：", source["source_id"])
    print(source["text"])
    for criterion in source["review_criteria"]:
        print("-", criterion)


 pilot_01_actions 来源： pilot_story_01
雨水顺着伞尖滴在门垫上。季禾把伞靠在墙边，低声说：我把文件放在柜子里了。唐悦拉开抽屉，里面只有一张空白纸。
- 雨滴、靠伞、拉开抽屉均由旁白保留
- 季禾台词保留低声
- 不混淆柜子与抽屉，保留只有一张空白纸

 pilot_02_three_roles 来源： pilot_story_02
许衡把登记簿推到桌子中央。罗薇问：这一页是谁撕掉的？店员说：我接班的时候就这样了。许衡说：先别翻下一页。三人停下了手里的动作。
- 罗薇、店员、许衡台词归属正确
- 保留推登记簿与三人停下动作
- 不解释谁撕掉了这一页

 pilot_03_inner_thought 来源： pilot_story_03
程墨看着手里的车票，日期是星期五。他心想，今天明明才星期四。他把车票折好放进口袋，没有把疑惑告诉身旁的人。
- 全部由旁白叙述，保留他心想
- 保留星期五与星期四
- 保留折好车票、放进口袋和没有告诉身旁的人

 pilot_04_no_dialogue 来源： pilot_story_04
午后，乔安在阳台整理花盆。最小的那盆土还是湿的，其余几盆已经干了。她把水壶放回原处，没有浇水。楼下传来一声自行车铃。
- 全部由旁白叙述，不新增对白
- 保留土壤湿干差异与没有浇水
- 保留一声自行车铃，不解释谁浇过水

 pilot_05_ambiguity 来源： pilot_story_05
值班员看着监控画面说：那个人又来了。白芷问：你认得他？值班员没有回答，只把画面暂停在电梯门打开的那一刻。画面里的人背对镜头，手里拿着一个纸袋。
- 不解释那个人是谁或纸袋里装了什么
- 保留没有回答和暂停画面的时刻
- 保留背对镜头与手里的一个纸袋


In [7]:
# 答案已由 agent 编写完整；你负责审阅写法并决定是否采用。
# 如需修改，重新运行本 Cell，再执行后续预览与检查。
ANSWERS = {
    "pilot_01_actions": """
[旁白][平静]雨水顺着伞尖滴在门垫上。
[旁白][平静]季禾把伞靠在墙边。
[季禾][低声]我把文件放在柜子里了。
[旁白][平静]唐悦拉开抽屉，里面只有一张空白纸。
""".strip(),
    "pilot_02_three_roles": """
[旁白][平静]许衡把登记簿推到桌子中央。
[罗薇][疑问]这一页是谁撕掉的？
[店员][平静]我接班的时候就这样了。
[许衡][平静]先别翻下一页。
[旁白][平静]三人停下了手里的动作。
""".strip(),
    "pilot_03_inner_thought": """
[旁白][平静]程墨看着手里的车票，日期是星期五。
[旁白][平静]他心想，今天明明才星期四。
[旁白][平静]他把车票折好放进口袋，没有把疑惑告诉身旁的人。
""".strip(),
    "pilot_04_no_dialogue": """
[旁白][平静]午后，乔安在阳台整理花盆。
[旁白][平静]最小的那盆土还是湿的，其余几盆已经干了。
[旁白][平静]她把水壶放回原处，没有浇水。
[旁白][平静]楼下传来一声自行车铃。
""".strip(),
    "pilot_05_ambiguity": """
[旁白][平静]值班员看着监控画面。
[值班员][平静]那个人又来了。
[白芷][疑问]你认得他？
[旁白][平静]值班员没有回答，只把画面暂停在电梯门打开的那一刻。
[旁白][平静]画面里的人背对镜头，手里拿着一个纸袋。
""".strip(),
}
print("已准备：", sum(bool(answer.strip()) for answer in ANSWERS.values()), "/", len(SOURCES))

已准备： 5 / 5


In [8]:
# 预览：修改 ANSWERS 后，先重跑答案 Cell，再从这里执行预览与检查。
for source in SOURCES:
    print("\n===", source["id"], "===")
    print("原文：", source["text"])
    print("答案：\n" + (ANSWERS.get(source["id"], "").strip() or "（尚未填写）"))


=== pilot_01_actions ===
原文： 雨水顺着伞尖滴在门垫上。季禾把伞靠在墙边，低声说：我把文件放在柜子里了。唐悦拉开抽屉，里面只有一张空白纸。
答案：
[旁白][平静]雨水顺着伞尖滴在门垫上。
[旁白][平静]季禾把伞靠在墙边。
[季禾][低声]我把文件放在柜子里了。
[旁白][平静]唐悦拉开抽屉，里面只有一张空白纸。

=== pilot_02_three_roles ===
原文： 许衡把登记簿推到桌子中央。罗薇问：这一页是谁撕掉的？店员说：我接班的时候就这样了。许衡说：先别翻下一页。三人停下了手里的动作。
答案：
[旁白][平静]许衡把登记簿推到桌子中央。
[罗薇][疑问]这一页是谁撕掉的？
[店员][平静]我接班的时候就这样了。
[许衡][平静]先别翻下一页。
[旁白][平静]三人停下了手里的动作。

=== pilot_03_inner_thought ===
原文： 程墨看着手里的车票，日期是星期五。他心想，今天明明才星期四。他把车票折好放进口袋，没有把疑惑告诉身旁的人。
答案：
[旁白][平静]程墨看着手里的车票，日期是星期五。
[旁白][平静]他心想，今天明明才星期四。
[旁白][平静]他把车票折好放进口袋，没有把疑惑告诉身旁的人。

=== pilot_04_no_dialogue ===
原文： 午后，乔安在阳台整理花盆。最小的那盆土还是湿的，其余几盆已经干了。她把水壶放回原处，没有浇水。楼下传来一声自行车铃。
答案：
[旁白][平静]午后，乔安在阳台整理花盆。
[旁白][平静]最小的那盆土还是湿的，其余几盆已经干了。
[旁白][平静]她把水壶放回原处，没有浇水。
[旁白][平静]楼下传来一声自行车铃。

=== pilot_05_ambiguity ===
原文： 值班员看着监控画面说：那个人又来了。白芷问：你认得他？值班员没有回答，只把画面暂停在电梯门打开的那一刻。画面里的人背对镜头，手里拿着一个纸袋。
答案：
[旁白][平静]值班员看着监控画面。
[值班员][平静]那个人又来了。
[白芷][疑问]你认得他？
[旁白][平静]值班员没有回答，只把画面暂停在电梯门打开的那一刻。
[旁白][平静]画面里的人背对镜头，手里拿着一个纸袋。


## 步骤 3A：格式与来源检查

5 条答案已完整准备。下面核对格式与来源；若后续修订引入空答案或格式错误，会指出具体 ID 和行号。

程序检查标签、正文、允许的角色、重复输入和与固定测试输入的完全重复。它不能自动证明事实完整或发现所有近似改写。事实与演绎标记由 agent 逐句核对，你审阅并决定是否采用。

In [9]:
LINE_PATTERN = re.compile(r"^\[([^\[\]\r\n]+)\]\[([^\[\]\r\n]+)\](\S.*)$")

def canonical_text(text):
    return re.sub(r"\s+", "", text)

def fingerprint(value):
    canonical = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def validate_pilot(sources, answers, fixed_cases):
    errors, records = [], []
    seen_ids, seen_texts = set(), set()
    eval_texts = {
        canonical_text(message["content"].removeprefix("改写下面的文本："))
        for case in fixed_cases
        for message in case["messages"]
        if message["role"] == "user"
    }
    if len(sources) != 5:
        errors.append("本教学阶段需要 5 条来源记录")
    if set(answers) != {source["id"] for source in sources}:
        errors.append("ANSWERS 的 ID 必须与 SOURCES 完全一致")
    for source in sources:
        identifier = source["id"]
        if not isinstance(identifier, str) or not identifier.strip() or identifier in seen_ids:
            errors.append(f"ID 缺失或重复：{identifier!r}")
        seen_ids.add(identifier)
        if not isinstance(source.get("source_id"), str) or not source["source_id"].strip():
            errors.append(f"{identifier}: 缺少 source_id")
        text = source["text"]
        if not isinstance(text, str) or not text.strip():
            errors.append(f"{identifier}: 原文为空")
            continue
        normalized = canonical_text(text)
        if normalized in seen_texts or normalized in eval_texts:
            errors.append(f"{identifier}: 原文重复，或与固定评估输入重复")
        seen_texts.add(normalized)
        answer = answers.get(identifier)
        if not isinstance(answer, str) or not answer.strip():
            errors.append(f"{identifier}: 尚未填写答案")
            continue
        answer = answer.strip()
        if re.search(r"</?think\b", answer, flags=re.I) or "```" in answer:
            errors.append(f"{identifier}: 答案不能含 think 标签或代码围栏")
        for number, line in enumerate(answer.splitlines(), 1):
            match = LINE_PATTERN.fullmatch(line)
            if not match or not match.group(2).strip() or not match.group(3).strip():
                errors.append(f"{identifier} 第 {number} 行: 需要 [角色][语气]正文，标签连续、正文非空")
            elif match.group(1) not in source["allowed_roles"]:
                errors.append(f"{identifier} 第 {number} 行: 不允许的角色 {match.group(1)!r}")
        records.append({"messages": [
            {"role": "system", "content": TTS_SYSTEM},
            {"role": "user", "content": "改写下面的文本：\n" + text},
            {"role": "assistant", "content": answer},
        ]})
    if errors:
        raise ValueError("数据检查未通过：\n- " + "\n- ".join(errors))
    return records

pilot_records = validate_pilot(SOURCES, ANSWERS, eval_cases)
print("格式与来源检查通过：", len(pilot_records), "条。仍需人工核对事实与演绎标记。")

格式与来源检查通过： 5 条。仍需人工核对事实与演绎标记。


## 步骤 3B：训练模板 token 长度

离线读取本地 processor，使用当前 ms-swift 的训练模板计算完整对话长度。包含模板内部的非思考前缀，不只是 assistant 正文。

每条必须不超过 512 tokens，超限会报错，不截断。训练入口建立时保持相同模板设置。模板内部前缀是否参与 loss，在训练阶段另行核对。

In [11]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["USE_HF"] = "1"
from swift.model import get_model_processor
from swift.template import get_template

unloaded_model, processor = get_model_processor(
    str(model.path),
    model_type=model.model_type,
    load_model=False,
    download_model=False,
    use_hf=True,
    device_map="cpu",
    model_kwargs={"local_files_only": True},
)
assert unloaded_model is None, "数据准备不应加载模型权重"
template = get_template(
    processor,
    template_type="qwen3_5",
    max_length=MAX_LENGTH,
    truncation_strategy="raise",
    enable_thinking=False,
)
template.set_mode("train")
print("已离线读取 processor 和训练模板；未加载模型权重。")

[INFO:swift] Setting torch_dtype: torch.bfloat16
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[INFO:swift] default_system: None
[INFO:swift] max_length: 512
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000


已离线读取 processor 和训练模板；未加载模型权重。


In [12]:
def measure_tokens(records, sources, active_template):
    report = []
    for source, record in zip(sources, records, strict=True):
        # encode 可能为内部模板添加前缀，传入副本保护人工答案。
        encoded = active_template.encode(copy.deepcopy(record))
        token_count = len(encoded["input_ids"])
        supervised_tokens = sum(label != -100 for label in encoded["labels"])
        if token_count > MAX_LENGTH or supervised_tokens == 0:
            raise ValueError(f"{source['id']}: 长度超限或没有可训练的 assistant tokens")
        report.append({"id": source["id"], "total_tokens": token_count, "supervised_tokens": supervised_tokens})
        print(source["id"], "总 tokens：", token_count, "/", MAX_LENGTH)
    return report

# 每次重新检查都重置人工复核状态，避免沿用旧答案的确认。
HUMAN_REVIEWED = False
# 重新构建，避免使用修改答案前的旧数据。
pilot_records = validate_pilot(SOURCES, ANSWERS, eval_cases)
length_report = measure_tokens(pilot_records, SOURCES, template)
CHECKED_STATE_SHA256 = fingerprint({"sources": SOURCES, "answers": ANSWERS, "max_length": MAX_LENGTH})
print("5 条训练模板长度检查通过。")

pilot_01_actions 总 tokens： 199 / 512
pilot_02_three_roles 总 tokens： 203 / 512
pilot_03_inner_thought 总 tokens： 181 / 512
pilot_04_no_dialogue 总 tokens： 198 / 512
pilot_05_ambiguity 总 tokens： 218 / 512
5 条训练模板长度检查通过。


## 步骤 3C：审阅决策并导出

agent 已按每条 `review_criteria` 核对答案，摘要见 `../datasets/pilot_review.md`。你审阅稿件并决定是否采用；同意后将下一 Cell 的 `HUMAN_REVIEWED` 改成 `True`，再运行导出。需要调整时说明偏好，由 agent 修订。

修改原文或答案后，重新执行答案、预览、格式检查和 token 检查，并重新审阅。导出仅保存检查过的数据。

In [14]:
# 你审阅并同意采用这 5 条答案后，改成 True；agent 的检查不代替你的决策。
HUMAN_REVIEWED = True

In [15]:
current_records = validate_pilot(SOURCES, ANSWERS, eval_cases)
current_state = fingerprint({"sources": SOURCES, "answers": ANSWERS, "max_length": MAX_LENGTH})
if current_state != CHECKED_STATE_SHA256:
    raise ValueError("原文、答案或长度上限发生变化，请重新运行 token 检查并做人工复核")
if HUMAN_REVIEWED is not True:
    raise ValueError("请先按每条检查项完成人工复核，再设置 HUMAN_REVIEWED = True")

output_dir = project_root / "datasets" / "processed" / (
    "pilot-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
)
output_dir.mkdir(parents=True, exist_ok=False)
for name, records in (("pilot.jsonl", current_records), ("sources.jsonl", SOURCES)):
    with (output_dir / name).open("x", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")
metadata = {
    "created_at": datetime.now(UTC).isoformat(),
    "stage": "five_sample_pilot",
    "sample_count": len(current_records),
    "prompt_ids": [source["id"] for source in SOURCES],
    "model_path": str(model.path),
    "max_length": MAX_LENGTH,
    "template": {"type": "qwen3_5", "mode": "train", "enable_thinking": False, "add_non_thinking_prefix": True, "loss_scale": "default", "truncation_strategy": "raise"},
    "baseline_dir": str(baseline_dir),
    "eval_prompts_sha256": prompts_fingerprint(eval_cases),
    "data_sha256": fingerprint(current_records),
    "checked_state_sha256": CHECKED_STATE_SHA256,
    "human_reviewed": True,
    "length_report": length_report,
    "packages": {name: version(name) for name in ("ms-swift", "transformers")},
}
(output_dir / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("教学样本：", output_dir / "pilot.jsonl")
print("来源记录：", output_dir / "sources.jsonl")
print("检查记录：", output_dir / "metadata.json")
print("下一步：扩充到 30 条训练数据和 6 条独立验证数据，再建立训练入口。")

教学样本： /home/muyi086/github/test_LLM/text-lora/datasets/processed/pilot-20261008T120902_009657Z/pilot.jsonl
来源记录： /home/muyi086/github/test_LLM/text-lora/datasets/processed/pilot-20261008T120902_009657Z/sources.jsonl
检查记录： /home/muyi086/github/test_LLM/text-lora/datasets/processed/pilot-20261008T120902_009657Z/metadata.json
下一步：扩充到 30 条训练数据和 6 条独立验证数据，再建立训练入口。


## 5 条通过后的安排

这 5 条可以作为训练集的起点，验证集另写独立故事。建议训练集为 26 条改写 + 4 条普通任务，验证集为 5 条改写 + 1 条普通任务；各类型分配见标注指南。

后续由 agent 编写并检查 `datasets/train.jsonl`、`datasets/val.jsonl` 及来源清单，完成来源分离、重复、格式和训练模板 token 长度检查，再建立 LoRA 训练入口。你负责审阅数据与决定是否继续。本 Notebook 到教学样本导出为止。